# Projeto 2 - Ambiente 3D
## Disciplina de Computação Gráfica

### Julia Cavallio Orlando - 14758721
### Miguel Rodrigues Tomazini - 14599300

### 1. Importação das Bibliotecas

In [1]:
import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math
from PIL import Image

from shader_s import Shader

### 2. Inicialização da Janela GLFW

In [2]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)

altura = 1040
largura = 1280

window = glfw.create_window(largura, altura, "Fuga do Estadio", None, None)

if (window == None):
    print("Falha ao criar a janela GLFW")
    glfw.terminate()
    
glfw.make_context_current(window)

### 3. Construção e Compilação dos Shaders

In [3]:
ourShader = Shader("vertex_shader.vs", "fragment_shader.fs")
ourShader.use()

program = ourShader.getProgram()

### 4. Interpretador de Modelos (.obj) e Texturas
Funções responsáveis por fazer o parse dos arquivos Wavefront `.obj`, lidando com triangulação de polígonos, filtragem por material (`target_material`) e carregamento de imagens para a memória da GPU.

In [4]:
glEnable(GL_TEXTURE_2D)
glHint(GL_LINE_SMOOTH_HINT, GL_DONT_CARE)
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)
glEnable(GL_LINE_SMOOTH)

global vertices_list
vertices_list = []    
global textures_coord_list
textures_coord_list = []

def load_model_from_file(filename):
    """Lê o arquivo .obj e extrai vértices, coordenadas UV e mapeamento de faces/materiais."""
    objects = {}
    vertices = []
    texture_coords = []
    faces = []
    material = None

    for line in open(filename, "r"):
        if line.startswith('#'): continue
        values = line.split()
        if not values: continue

        if values[0] == 'v':
            vertices.append(values[1:4])
        elif values[0] == 'vt':
            texture_coords.append(values[1:3])
        elif values[0] in ('usemtl', 'usemat'):
            material = values[1]
        elif values[0] == 'f':
            face = []
            face_texture = []
            for v in values[1:]:
                w = v.split('/')
                face.append(int(w[0]))
                if len(w) >= 2 and len(w[1]) > 0:
                    face_texture.append(int(w[1]))
                else:
                    face_texture.append(0)
            faces.append((face, face_texture, material))

    model = {'vertices': vertices, 'texture': texture_coords, 'faces': faces}
    return model

def load_texture_from_file(texture_id, img_textura):
    """Carrega uma imagem e a vincula a um buffer de textura 2D do OpenGL."""
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)

    
    img = Image.open(img_textura)
    # Converte imagens de 16-bit para 8-bit usando numpy
    if img.mode in ('I;16', 'I;16B', 'I'):
        arr = np.array(img, dtype=np.uint16)
        arr = (arr >> 8).astype(np.uint8)
        img = Image.fromarray(arr, mode='L')
    img = img.convert("RGB")


    img_width = img.size[0]
    img_height = img.size[1]
    image_data = img.tobytes("raw", "RGB", 0, -1)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, img_width, img_height, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)

def circular_sliding_window_of_three(arr):
    """Garante a triangulação de polígonos com mais de 3 lados (ex: quads)."""
    if len(arr) == 3:
        return arr
    circular_arr = arr + [arr[0]]
    result = []
    for i in range(len(circular_arr) - 2):
        result.extend(circular_arr[i:i+3])
    return result

global numberTextures
numberTextures = 0

def load_obj_and_texture(objFile, texturesList, target_material=None):
    """Integra vértices e coordenadas UV, filtrando opcionalmente por material."""
    modelo = load_model_from_file(objFile)
    verticeInicial = len(vertices_list)
    
    for face in modelo['faces']:
        # Se um target_material for definido, pula as faces que não o possuem (útil para sub-meshes)
        if target_material is not None and face[2] != target_material:
            continue
            
        for vertice_id in circular_sliding_window_of_three(face[0]):
            vertices_list.append(modelo['vertices'][vertice_id - 1])
            
        for texture_id in circular_sliding_window_of_three(face[1]):
            u = float(modelo['texture'][texture_id - 1][0])
            v = float(modelo['texture'][texture_id - 1][1])
            textures_coord_list.append([u, v])
        
    verticeFinal = len(vertices_list)
    
    global numberTextures
    tex_start_id = numberTextures
    for i in range(len(texturesList)):
        load_texture_from_file(numberTextures, texturesList[i])
        numberTextures += 1
    
    return verticeInicial, verticeFinal - verticeInicial, tex_start_id

### 5. Registro e Instanciação dos Assets
Cria um dicionário em memória que mapeia os objetos aos seus respectivos VBOs, permitindo o reaproveitamento de malhas e controle modular das texturas (incluindo tilling através de `tex_scale_u` e `tex_scale_v`).

In [5]:
game_assets = {}

def register_model(name, obj_path, tex_path, tex_scale_u=1.0, tex_scale_v=1.0, target_material=None):
    """Registra o modelo no dicionário de assets, pré-processando texturas e UV mapping."""
    # Se tex_scale_v não for definido explicitamente, mantém a proporção uniforme com o eixo U
    if tex_scale_v == 1.0 and tex_scale_u != 1.0:
        tex_scale_v = tex_scale_u
        
    start, count, tex_id = load_obj_and_texture(obj_path, [tex_path], target_material)
    
    game_assets[name] = {
        'start': start, 
        'count': count, 
        'tex_id': tex_id, 
        'tex_scale_u': float(tex_scale_u), 
        'tex_scale_v': float(tex_scale_v)
    }

# --- REGISTRO DOS MODELOS ---

# AMBIENTE INTERNO
register_model('floor_int', 'objects/floor/floor.obj', 'objects/floor/floor_internal.jpg', tex_scale_u=10.0, tex_scale_v=10.0)
# No bloco de Registro de Modelos:

register_model('room_parede1', 'objects/room/room.obj', 'objects/room/texture.jpg',  target_material='parede1', tex_scale_u=5.0)
register_model('room_parede2', 'objects/room/room.obj', 'objects/room/textura2.jpg', target_material='parede2', tex_scale_u=5.0)
register_model('room_teto',    'objects/room/room.obj', 'objects/room/textura3.jpg', target_material='teto')
register_model('room_chao',    'objects/room/room.obj', 'objects/room/texture.jpg',  target_material='chao', tex_scale_u=5.0)

register_model('bench', 'objects/bench/eb_hipster_bench_01.obj', 'objects/bench/eb_hipster_bench_01_c.tga')
register_model('ceiling_fan', 'objects/ceiling_fan/ceiling_fan.obj', 'objects/ceiling_fan/texture.jpg')
register_model('treadmill', 'objects/treadmill/uploads_files_48444_Treadmill.obj', 'objects/treadmill/Treadmill.png')
register_model('table_tennis', 'objects/Table tennis/table_tennis.obj', 'objects/Table tennis/TT_texture.jpg')
register_model('ab_machine', 'objects/ab/uploads_files_2717617_Ab_King.obj', 'objects/ab/Ab_King_Material _2_AO.png')

# AMBIENTE EXTERNO
register_model('skybox', 'objects/skybox/skybox.obj', 'objects/skybox/texture.png')
register_model('floor_ext', 'objects/floor/floor.obj', 'objects/floor/floor_external.jpg', tex_scale_u=20.0, tex_scale_v=20.0)
register_model('field', 'objects/field/field.obj', 'objects/field/texture.png')
register_model('ball', 'objects/ball/ball.obj', 'objects/ball/texture.png')
register_model('net', 'objects/net/net.obj', 'objects/net/texture.jpg')
register_model('footballer', 'objects/footballer-3d-model/FootballerRigged-bl.obj', 'objects/footballer-3d-model/Diniz (5).png')

# ARQUIBANCADA (Uso de Sub-Meshes para aplicar texturas distintas no mesmo modelo 3D)
register_model(
    name='arquibancada_madeira', 
    obj_path='objects/bleacher/bleacher.obj', 
    tex_path='objects/bleacher/textures/Wood_BaseColor.png', 
    target_material='Wood'
)

register_model(
    name='arquibancada_metal', 
    obj_path='objects/bleacher/bleacher.obj', 
    tex_path='objects/bleacher/textures/Metal_BaseColor.png', 
    target_material='Metal'
)

def draw_model(name, angle=0.0, r_x=0.0, r_y=1.0, r_z=0.0, t_x=0.0, t_y=0.0, t_z=0.0, s_x=1.0, s_y=1.0, s_z=1.0, escala=1.0):
    """Aplica a matriz Model (Transformações) e envia os dados uniformes ao Shader para desenhar o objeto."""
    asset = game_assets[name]
    
    # Matriz Model (Escala global aplicada aos eixos)
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x * escala, s_y * escala, s_z * escala)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
    
    # Envio das Escalas UV para o Vertex Shader (Controla o Tiling da Textura)
    loc_tex_scale_u = glGetUniformLocation(program, "tex_scale_u")
    glUniform1f(loc_tex_scale_u, asset['tex_scale_u'])
    
    loc_tex_scale_v = glGetUniformLocation(program, "tex_scale_v")
    glUniform1f(loc_tex_scale_v, asset['tex_scale_v'])
    
    glBindTexture(GL_TEXTURE_2D, asset['tex_id'])
    glDrawArrays(GL_TRIANGLES, asset['start'], asset['count'])

/tmp/ipykernel_12029/4179771956.py:60: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(arr, mode='L')


### 6. Configuração dos Buffers da GPU (VBO e VAO)
Envia as listas globais construídas pelas funções anteriores diretamente para a memória da Placa de Vídeo.

In [6]:
buffer_VBO = glGenBuffers(2)

# Buffer de Vértices (Coordenadas X, Y, Z)
vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
vertices['position'] = vertices_list
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)
loc_vertices = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc_vertices)
glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, stride, offset)

# Buffer de Texturas (Coordenadas U, V)
textures = np.zeros(len(textures_coord_list), [("position", np.float32, 2)])
textures['position'] = textures_coord_list
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
glBufferData(GL_ARRAY_BUFFER, textures.nbytes, textures, GL_STATIC_DRAW)
stride = textures.strides[0]
offset = ctypes.c_void_p(0)
loc_texture_coord = glGetAttribLocation(program, "texture_coord")
glEnableVertexAttribArray(loc_texture_coord)
glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, stride, offset)

### 7. Controles de Estado, Input e Câmera
Define a captura assíncrona do mouse (para olhar ao redor calculando ângulos de Euler) e variáveis globais que determinam a lógica física do mundo.

In [7]:
# Variáveis de Câmera (FPS Style)
cameraPos   = glm.vec3(0.0, 1.0, 0.0)
cameraFront = glm.vec3(0.0, 0.0, -1.0)
cameraUp    = glm.vec3(0.0, 1.0, 0.0)

# Variáveis da Lógica da Bola
ball_pos = glm.vec3(0.0, 0.0, -30.0) 
ball_vel = glm.vec3(0.0, 0.0, 0.0)   
ball_rotation = 0.0                 

# Controle de Tempo (Delta Time para consistência de framerate)
deltaTime = 0.0
lastFrame = 0.0

# Variáveis de Mouse/Visão
firstMouse = True
yaw = -90.0 
pitch = 0.0
lastX = largura / 2
lastY = altura / 2
fov = 45.0
polygonal_mode = False

# Dicionário de Inputs contínuos
teclas = {}
ball_scale = 2.0
player_pos = glm.vec3(0.0, 0.0, -15.0)
game_over = False

# Fatores de Física (Gravidade e Pulo)
vel_y = 0.0
is_jumping = False
gravity = -25.0
jump_force = 10.0

# Variáveis Interativas (Ventilador)
fan_rotation = 0.0
fan_speed = 150.0

def key_event(window, key, scancode, action, mods):
    """Callback para registrar teclas pressionadas de forma contínua ou ativar flags (ex: Modo Malha)."""
    global polygonal_mode, teclas
    
    if action == glfw.PRESS:
        teclas[key] = True
        if key == glfw.KEY_ESCAPE:
            glfw.set_window_should_close(window, True)
        if key == glfw.KEY_P:
            polygonal_mode = not polygonal_mode
    elif action == glfw.RELEASE:
        teclas[key] = False

def mouse_callback(window, xpos, ypos):
    """Callback do mouse converte deslocamento em graus para gerar o vetor de direção (cameraFront)."""
    global cameraFront, lastX, lastY, firstMouse, yaw, pitch
   
    if firstMouse:
        lastX = xpos
        lastY = ypos
        firstMouse = False

    xoffset = xpos - lastX
    yoffset = lastY - ypos
    lastX = xpos
    lastY = ypos

    sensitivity = 0.1
    xoffset *= sensitivity
    yoffset *= sensitivity
    yaw += xoffset
    pitch += yoffset

    # Impede que a tela vire de cabeça para baixo
    if pitch > 89.0: pitch = 89.0
    if pitch < -89.0: pitch = -89.0

    front = glm.vec3()
    front.x = glm.cos(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    front.y = glm.sin(glm.radians(pitch))
    front.z = glm.sin(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    cameraFront = glm.normalize(front)

glfw.set_key_callback(window, key_event)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)

### 8. Matrizes de Visualização (MVP)
Geração das matrizes responsáveis por posicionar os objetos (Model), posicionar a câmera (View) e garantir a profundidade/perspectiva (Projection).

In [8]:
def model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    """Retorna a matriz de transformação do objeto no espaço de mundo."""
    angle = math.radians(angle)
    matrix_transform = glm.mat4(1.0)
    matrix_transform = glm.translate(matrix_transform, glm.vec3(t_x, t_y, t_z))    
    if angle != 0:
        matrix_transform = glm.rotate(matrix_transform, angle, glm.vec3(r_x, r_y, r_z))
    matrix_transform = glm.scale(matrix_transform, glm.vec3(s_x, s_y, s_z))
    return np.array(matrix_transform)

def view():
    """Retorna a matriz View baseada na posição atual e direção da câmera."""
    global cameraPos, cameraFront, cameraUp
    mat_view = glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp)
    return np.array(mat_view)

def projection():
    """Retorna a matriz de projeção em perspectiva."""
    global altura, largura
    # Far plane definido em 1000.0 para suportar renderização do Skybox de grande escala
    mat_projection = glm.perspective(glm.radians(fov), largura/altura, 0.1, 1000.0)
    return np.array(mat_projection)

### 9. Lógica de Colisão AABB
Mecanismo de colisão do tipo *Axis-Aligned Bounding Box*, que impede que a câmera atravesse malhas opacas do cenário baseando-se em limites geométricos estáticos.

In [9]:
def check_collision(x, z, radius = 1.0):
    """Verifica se a coordenada (x, z) desejada sobrepõe as geometrias sólidas."""
    
    # Limites externos (Skybox e limites do Estádio)
    if x < -30.0 or x > 30.0 or z < -60.0 or z > 60.0:
        return True
        
    # --- COLISÕES DO VESTIÁRIO (Room) ---
    # A sala base está em s_x=30 e s_z=15 e posicionada em t_z=15.
    # A espessura das paredes está em 0.05 da escala normalizada.
    
    # 1. Parede Frontal (onde tem a porta de passagem) - eixo Z
    if 0.0 - radius < z < 0.75 + radius:
        # Vão da porta (Caminho livre caso X esteja entre -6.0 e 6.0)
        if x < -6.0 + radius or x > 6.0 - radius:
            return True 
            
    # 2. Paredes internas laterais e fundo (Verificadas apenas se a entidade entrou na sala Z >= 0.75)
    if z >= 0.75 - radius and z <= 30.0:
        if z > 29.25 - radius:  # Parede Fundo
            return True
        if x < -28.5 + radius:  # Parede Esquerda
            return True
        if x > 28.5 - radius:   # Parede Direita
            return True
            
    return False

### 10. Loop Principal de Execução
Mantém a janela aberta e calcula iterativamente a física (mecânica da bola, gravidade do pulo e tracking do inimigo), atualizando as entradas do usuário e emitindo comandos de renderização ordenados por profundidade.

In [10]:
glfw.show_window(window)
glEnable(GL_DEPTH_TEST) # Habilita Z-Buffer

lastFrame = glfw.get_time()

# Vetores auxiliares para a rotação realista da bola
rot_axis_x = 0.0
rot_axis_z = 1.0 
ball_rotation = 0.0

while not glfw.window_should_close(window):
    currentFrame = glfw.get_time()
    deltaTime = currentFrame - lastFrame
    lastFrame = currentFrame

    glfw.poll_events()
    
    # =========================================
    # FÍSICA DO JOGADOR (CÂMERA)
    # =========================================

    # --- Pulo e Gravidade (Integração de Euler) ---
    # Adiciona a força da gravidade continuamente e testa limite do piso em y=1.85 (Altura do observador)
    if glfw.get_key(window, glfw.KEY_SPACE) == glfw.PRESS and not is_jumping:
        vel_y = jump_force
        is_jumping = True
        
    vel_y += gravity * deltaTime
    cameraPos.y += vel_y * deltaTime
    
    if cameraPos.y <= 1.85:
        cameraPos.y = 1.85
        vel_y = 0.0
        is_jumping = False

    # --- Lógica de Movimentação Contínua (WASD) ---
    # Restringe o vetor forward (front_xz) apenas ao plano chão, evitando perda de velocidade ao olhar para baixo
    cameraSpeed = 10.0 * deltaTime
    new_pos = glm.vec3(cameraPos)
    front_xz = glm.normalize(glm.vec3(cameraFront.x, 0.0, cameraFront.z))

    if teclas.get(glfw.KEY_W):
        new_pos += cameraSpeed * front_xz
    if teclas.get(glfw.KEY_S):
        new_pos -= cameraSpeed * front_xz
    if teclas.get(glfw.KEY_A):
        new_pos -= glm.normalize(glm.cross(front_xz, cameraUp)) * cameraSpeed
    if teclas.get(glfw.KEY_D):
        new_pos += glm.normalize(glm.cross(front_xz, cameraUp)) * cameraSpeed
        
    # Aplicação separada de colisão por eixos, permitindo deslizar caso bata de raspão na parede
    if not check_collision(new_pos.x, cameraPos.z):
        cameraPos.x = new_pos.x
    if not check_collision(cameraPos.x, new_pos.z):
        cameraPos.z = new_pos.z


    # =========================================
    # FÍSICA DE INTERAÇÃO COM A BOLA
    # =========================================
    raio_bola = ball_scale / 2.0
    dist_player_ball = glm.length(glm.vec3(cameraPos.x, 0.0, cameraPos.z) - glm.vec3(ball_pos.x, 0.0, ball_pos.z))
    
    # Se o raio da bola + raio da colisão do jogador encostarem
    if dist_player_ball < raio_bola + 1.0: 
        # Vetor Push (empurrão originado das costas da câmera para a bola)
        push_dir = glm.normalize(glm.vec3(ball_pos.x, 0.0, ball_pos.z) - glm.vec3(cameraPos.x, 0.0, cameraPos.z))
        
        push_strength = 15.0 
        ball_vel = push_dir * push_strength
        
    speed = glm.length(ball_vel)
    if speed > 0.01:
        new_ball_x = ball_pos.x + ball_vel.x * deltaTime
        new_ball_z = ball_pos.z + ball_vel.z * deltaTime
        
        # Fator Bounciness de perda cinética (0.8 retém 80% da energia no impacto com a parede)
        bounce = 0.8 
        
        # Colisão dinâmica da bola com os limites do mapa
        if check_collision(new_ball_x, ball_pos.z, radius=raio_bola):
            ball_vel.x *= -bounce 
        else:
            ball_pos.x = new_ball_x
            
        if check_collision(ball_pos.x, new_ball_z, radius=raio_bola):
            ball_vel.z *= -bounce
        else:
            ball_pos.z = new_ball_z
            
        # Fator Fricção da Grama
        friction = 2.0 
        ball_vel -= ball_vel * friction * deltaTime
        
        # Conversão de distância angular para giro tridimensional visual da malha (.obj)
        ball_rotation += (speed * deltaTime / raio_bola) * (180.0 / math.pi)
        rot_axis_x = ball_vel.z / speed
        rot_axis_z = -ball_vel.x / speed

    # =========================================
    # EVENTOS DE AMBIENTE (TECLAS E IA)
    # =========================================
    # Escala da Bola Interativa
    if teclas.get(glfw.KEY_E):
        ball_scale += 0.5 * deltaTime
    if teclas.get(glfw.KEY_Q):
        ball_scale = max(0.1, ball_scale - 0.5 * deltaTime)
    
    # Velocidade Variável e Reverso dos Ventiladores
    if glfw.get_key(window, glfw.KEY_RIGHT) == glfw.PRESS:
        fan_speed += 100.0 * deltaTime
    if glfw.get_key(window, glfw.KEY_LEFT) == glfw.PRESS:
        fan_speed -= 100.0 * deltaTime
        
    fan_rotation += fan_speed * deltaTime

    # Lógica Pursuit (O Jogador IA segue a câmera se ela aproximar)
    if not game_over:
        dist_vec = cameraPos - player_pos
        dist_vec.y = 0 
        distance = glm.length(dist_vec)
        
        if distance < 1.0:
            game_over = True
            print("GAME OVER! O jogador te alcançou.")
        elif distance < 30.0:  
            direction = glm.normalize(dist_vec)
            player_pos += direction * 3.0 * deltaTime

    # =========================================
    # RENDERIZAÇÃO DA CENA
    # =========================================
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    glClearColor(0.2, 0.2, 0.2, 1.0)
    
    if polygonal_mode:
        glPolygonMode(GL_FRONT_AND_BACK, GL_LINE)
    else:
        glPolygonMode(GL_FRONT_AND_BACK, GL_FILL)

    # --- VESTIÁRIO (Sala e adereços) ---
    # No Loop Principal (Renderização):
    draw_model('room_parede1', t_z=15, t_y=0, s_x=30, s_y=5, s_z=15)
    draw_model('room_parede2', t_z=15, t_y=0, s_x=30, s_y=5, s_z=15)
    draw_model('room_teto',    t_z=15, t_y=0, s_x=30, s_y=5, s_z=15)
    draw_model('room_chao',    t_z=15, t_y=0, s_x=30, s_y=5, s_z=15)


    draw_model('floor_int', t_z=15, t_y=0.01, s_x=60, s_z=30, escala=0.5)

    draw_model('bench', t_x=-10.0, t_y=0.0, t_z=15.0, escala=0.015)
    draw_model('bench', t_x=10.0, t_y=0.0, t_z=15.0, escala=0.015)
    draw_model('bench', t_x=20.0, t_y=0.0, t_z=15.0, escala=0.015)
    draw_model('bench', t_x=-20.0, t_y=0.0, t_z=15.0, escala=0.015)

    for i in range(11):
        draw_model('treadmill', t_x=-25.0 + i*5.0, t_y=0.0, t_z=27.0, escala=0.03)

    draw_model('table_tennis', t_x=0.0, t_y=0.8, t_z=15.0, escala=0.015)
    
    for i in range(5):
        draw_model('ab_machine', t_x=-25.0 + i * 4.0, t_y=0.0, t_z=5.0, angle=-180.0, r_y=1.0, escala=0.024)
    for i in range(5):
        draw_model('ab_machine', t_x=25.0 - i * 4.0, t_y=0.0, t_z=5.0, angle=-180.0, r_y=1.0, escala=0.024)
    
    draw_model('ceiling_fan', t_y=1.0, t_z=15, angle=fan_rotation, r_y=1, escala=2.5)
    draw_model('ceiling_fan', t_x=15, t_y=1.0, t_z=15, angle=fan_rotation, r_y=1, escala=2.5)
    draw_model('ceiling_fan', t_x=-15, t_y=1.0, t_z=15, angle=fan_rotation, r_y=1, escala=2.5)

    # --- ESTÁDIO EXTERNO (Campos, Traves e Chão) ---
    # O Skybox ancora-se na coordenada da câmera para simular o infinito
    draw_model('skybox', t_x=cameraPos.x, t_y=cameraPos.y, t_z=cameraPos.z, escala=500.0)
    draw_model('field', t_y=0.01, t_z=-30, s_x=60, s_z=60, escala=0.5)
    draw_model('floor_ext', t_z=-50, s_x=160, s_z=200, escala=0.5)
    
    draw_model('net', t_x=29.4, t_z=-30, angle=180.0, r_y=1.0)
    draw_model('net', t_x=-29.4, t_z=-30)

    # --- ENTIDADES MÓVEIS (Jogador e Bola) ---
    draw_model('footballer', t_x=2.0, t_y=0.0, t_z=-28.0, angle=180.0, r_y=1.0, escala=0.3)

    draw_model('ball', 
        t_x=ball_pos.x, 
        t_y=(ball_scale/10), 
        t_z=ball_pos.z, 
        escala=ball_scale,
        angle=ball_rotation, 
        r_x=rot_axis_x, 
        r_y=0.0, 
        r_z=rot_axis_z
    )

    # --- ESTRUTURA DA ARQUIBANCADA ---
    # Desenhada com iteração em múltiplas instâncias para cercar o campo. 
    # Utiliza as Sub-Meshes (Metal e Madeira) registradas anteriormente.
    for i in range(6):
        # Lateral Esquerda
        draw_model('arquibancada_madeira', t_x=35, t_z=(-10*i)-5, s_y=0.75, escala=2.5, angle=-90.0, r_y=1.0)
        draw_model('arquibancada_metal',   t_x=35, t_z=(-10*i)-5, s_y=0.75, escala=2.5, angle=-90.0, r_y=1.0)
        # Lateral Direita
        draw_model('arquibancada_madeira', t_x=-35, t_z=(-10*i)-5, s_y=0.75, escala=2.5, angle=90.0, r_y=1.0)
        draw_model('arquibancada_metal',   t_x=-35, t_z=(-10*i)-5, s_y=0.75, escala=2.5, angle=90.0, r_y=1.0)

    for i in range(7):
        # Fundos
        draw_model('arquibancada_madeira', t_x=(10*i)-30, t_z=-65, s_y=0.75, escala=2.5)
        draw_model('arquibancada_metal',   t_x=(10*i)-30, t_z=-65, s_y=0.75, escala=2.5)

    # --- COMMIT DAS MATRIZES --- 
    # Atualiza as matrizes de View e Projection unicamente no final do frame antes de trocar o buffer
    mat_view = view()
    loc_view = glGetUniformLocation(program, "view")
    glUniformMatrix4fv(loc_view, 1, GL_TRUE, mat_view)

    mat_projection = projection()
    loc_projection = glGetUniformLocation(program, "projection")
    glUniformMatrix4fv(loc_projection, 1, GL_TRUE, mat_projection)    
    
    glfw.swap_buffers(window)

glfw.terminate()

GAME OVER! O jogador te alcançou.
